## Project - FPN Search - Generate FPN Questions

To be used for quality evaluation of models


In [1]:
import os
import json
import random

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

from datetime import datetime
import re
from huggingface_hub import HfApi, CommitOperationAdd
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

search_client = None
openai_api_key = None
openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [2]:
def GetOpenAIKey():
    openai_api_key = os.getenv('OPENAI_API_KEY')

    if openai_api_key:
        print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
    else:
        print("OpenAI API Key not set")
    return openai_api_key

In [3]:
system_message = r"""
**Prompt for Agent: Create Medical Reference Evaluation Questions**  
You are tasked with generating medical reference questions using the FP Notebook database. These questions are designed to evaluate and compare different large language models (LLMs) for their ability to answer clinical reference queries using information retrieved from the FP Notebook (via retrieval-augmented generation, or RAG).  
**Guidelines:**

- Focus on practical, clinically relevant questions that a healthcare provider might ask when using the FP Notebook as a quick reference.  
- Each question should be answerable in a single, evidence-based paragraph, suitable for brief clinical use.  
- Ensure every question is clearly grounded in medical scope covered by the FP Notebook.
- Avoid questions that require opinion, speculation, or information not present in the database.
- Format: Clear clinical question, appropriate for a one-shot answer (e.g., “What is the recommended first-line treatment for community-acquired pneumonia in adults?”).
- Variety: Include diagnosis, treatment, medication dosing, contraindications, differential diagnosis, and follow-up.
- Avoid ambiguous scenarios; questions should be answerable with information present in the FP Notebook.

**Sample Questions:**
- What are the diagnostic criteria for type 2 diabetes mellitus?
- Outline the management of acute asthma exacerbation in adults.
- List contraindications to beta blocker therapy.
- What is the typical dosing for amoxicillin in pediatric otitis media?

**Judging Criteria for LLM-Generated Answers:**  
When evaluating LLM responses, use these criteria:

1. **Medical Accuracy:** Is the answer clinically correct and free of factual errors?
2. **Database Faithfulness:** Is the answer strictly derived from information found in the FP Notebook database?
3. **Clinical Usefulness:** Would a provider find the response actionable and appropriately focused for quick reference?
4. **Completeness:** Does the answer sufficiently address the full scope of the question without unnecessary detail?
5. **Conciseness:** Is the answer a single, well-structured paragraph suited for quick clinical review?
6. **Clarity:** Is the language clear, precise, and free of ambiguity?

**Instructions for Agent:**  
Generate a diverse set of clinical reference questions following the guidelines above.  Ensure a diverse array of questions covering in total topics from  the nearly 30 specialty based books in fpnotebook (e.g. cardiology, dermatology, endocrinology).   Ensure each question is grounded in the FP Notebook’s scope and requires no outside information to answer accurately.  

"""

In [4]:
include_history = False
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    if not include_history:
        history = []        
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [5]:
def chat_json(message):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": message}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    output = json.loads(result)
    return output

In [7]:
openai_api_key = GetOpenAIKey()
jsonOutput = chat_json("Generate 100 questions following the guidelines in system instructions.  Format to save in a file as a json array of strings. ")

with open('sample_questions.json', 'w') as file:
    json.dump(jsonOutput, file, indent=4)

OpenAI API Key exists and begins sk-proj-
